# Lesson 16 Interactive Lab
## OpenAI Embeddings, Persistent ChromaDB, Top-K Retrieval, and Chat Memory

This notebook lets you generate live embeddings, store them in ChromaDB, compare Top-K retrieval, merge context, and understand session-state chat memory.

# Install
```bash
pip install openai chromadb numpy pandas matplotlib scikit-learn
```

In [12]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [13]:
api_key = os.getenv("OPENAI_API_KEY")
print("Loaded:", api_key[:5] + "..." + api_key[-4:] if api_key else "No key found")

Loaded: No key found


In [3]:
from openai import OpenAI
client = OpenAI(api_key=api_key) if api_key else None

In [4]:
chunks = [
    "Full-time employees receive 15 paid vacation days every calendar year.",
    "Vacation requests should be submitted to the manager at least two weeks in advance.",
    "Unused vacation days do not carry forward into the next calendar year.",
    "Employees may work remotely up to two days per week with manager approval.",
    "Remote employees must remain available during core hours from 10 AM to 3 PM.",
    "Passwords must not be shared and confidential files must be stored in approved systems.",
    "Security incidents must be reported to the IT team immediately."
]
metadata=[
 {"source":"employee_handbook.pdf","section":"Vacation Policy","chunk_index":0},
 {"source":"employee_handbook.pdf","section":"Vacation Policy","chunk_index":1},
 {"source":"employee_handbook.pdf","section":"Vacation Policy","chunk_index":2},
 {"source":"employee_handbook.pdf","section":"Remote Work","chunk_index":3},
 {"source":"employee_handbook.pdf","section":"Remote Work","chunk_index":4},
 {"source":"employee_handbook.pdf","section":"Security","chunk_index":5},
 {"source":"employee_handbook.pdf","section":"Security","chunk_index":6}
]
display(pd.DataFrame({"chunk":chunks,"section":[m["section"] for m in metadata]}))

,chunk,section
0,Full-time employees receive 15 paid vacation d...,Vacation Policy
1,Vacation requests should be submitted to the m...,Vacation Policy
2,Unused vacation days do not carry forward into...,Vacation Policy
3,Employees may work remotely up to two days per...,Remote Work
4,Remote employees must remain available during ...,Remote Work
5,Passwords must not be shared and confidential ...,Security
6,Security incidents must be reported to the IT ...,Security


In [5]:
EMBEDDING_MODEL="text-embedding-3-small"
def get_embedding(text):
    return client.embeddings.create(model=EMBEDDING_MODEL,input=text).data[0].embedding

In [6]:
if client:
    vector=get_embedding(chunks[0])
    print("Dimensions:",len(vector))
    print(vector[:15])

In [7]:
if client:
    arr=np.array(vector)
    plt.figure(figsize=(10,4))
    plt.plot(arr[:100])
    plt.title("First 100 Embedding Dimensions")
    plt.grid(True)
    plt.show()

In [8]:
chunk_embeddings=[get_embedding(c) for c in chunks] if client else []
print("Embedded:",len(chunk_embeddings))

Embedded: 0


In [9]:
import chromadb
db=chromadb.PersistentClient(path="./lesson16_chroma_db")
collection=db.get_or_create_collection(name="lesson16_knowledge")

In [10]:
if chunk_embeddings:
    collection.upsert(
        ids=[f"chunk_{i:03d}" for i in range(len(chunks))],
        documents=chunks, embeddings=chunk_embeddings, metadatas=metadata
    )
print("Count:",collection.count())

Count: 0


In [11]:
print(json.dumps(collection.peek(),indent=2))

TypeError: Object of type ndarray is not JSON serializable

In [ ]:
question="What is the annual leave allowance?"
if client:
    qvec=get_embedding(question)
    results=collection.query(query_embeddings=[qvec],n_results=3)
    print(json.dumps(results,indent=2))

In [ ]:
def top_k(question,k):
    qvec=get_embedding(question)
    return collection.query(query_embeddings=[qvec],n_results=k)

if client:
    for k in [1,3,5]:
        r=top_k("Explain the vacation policy.",k)
        print("\nTOP-K =",k)
        for d in r["documents"][0]: print("-",d)

In [ ]:
if client:
    r=top_k("Explain the vacation policy.",3)
    context="\n\n".join(r["documents"][0])
    print(context)

In [ ]:
prompt=f"""You are a document-grounded assistant.
Use only the context below.
If unsupported, say: I do not know based on the provided documents.

CONTEXT:
{context}

QUESTION:
Explain the vacation policy.

ANSWER:"""
print(prompt)

In [ ]:
if client:
    response=client.responses.create(
        model="gpt-5-mini",
        instructions="Answer only from the supplied context. If unsupported, say you do not know based on the provided documents.",
        input=prompt
    )
    print(response.output_text)

# Session State Concept
In Streamlit:
```python
if "messages" not in st.session_state:
    st.session_state.messages = []
```

In [ ]:
messages=[]
messages.append({"role":"user","content":"How many vacation days do I get?"})
messages.append({"role":"assistant","content":"You receive 15 paid vacation days."})
for m in messages: print(m["role"],":",m["content"])

# Reflection
1. Why store embeddings persistently?
2. What does Top-K control?
3. Why can too much context be harmful?
4. Why does Streamlit need session state?